# Import libraries

In [244]:
from pathlib import Path
import pandas as pd
import json
import logging
import re

# Logging Setup

In [245]:
# Current working directory
BASE_PATH = Path.cwd().parent

DATA_PATH = BASE_PATH / "data" / "app_logs_7days.jsonl"
LOG_PATH = BASE_PATH / "pipeline" / "logs" / "pipeline.log"
QUARANTINE_PATH = BASE_PATH / "pipeline" / "outputs" / "malformed_records.jsonl"

In [246]:
logger = logging.getLogger("DataProfiling")
logger.setLevel(logging.INFO)

# Clear existing handlers if re-running the notebook
if logger.hasHandlers():
    logger.handlers.clear()

file_handler = logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8")
formatter = logging.Formatter(
	fmt="%(asctime)s - %(levelname)s - %(message)s",
	datefmt="%Y-%m-%d %H:%M:%S"
)
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

# Data Loading

Note: Here I used **Lazy file streaming**. Why I think that's matter: 
- Python processes files sequentially line-by-line rather than loading entire datasets into RAM
- This ensures that memory usage stays contanst O(1), no matter what size our `.jsonl` file is 

In [247]:
success_count = 0
error_count = 0
valid_records = []

logger.info(f"Starting data profiling")
with open(DATA_PATH, mode="r", encoding="utf-8") as f_in, \
	open(QUARANTINE_PATH, mode="w", encoding="utf-8") as f_err: # overwrite mode
		# Stream line-by-line
		for line_num, line in enumerate(f_in, start=1):
			raw_line = line.strip() # <class 'str'>
		   
			if not raw_line:
				continue 
			
			try:
				record = json.loads(raw_line) # <class 'dict'>
				success_count += 1
				valid_records.append(record)
			
			except json.JSONDecodeError as e:
				error_count += 1
				logger.warning(f"Line {line_num} is malformed: {e}")
				
				malformed_record = {
					"source_file": str(DATA_PATH),
					"line_number": line_num,
					"raw_line": raw_line,
					"error_message": str(e)
				}
				f_err.write(json.dumps(malformed_record) + "\n")

logger.info(f"Data profiling completed. Successfully processed {success_count} records. Found {error_count} malformed records.")

# EDA

In [248]:
df = pd.DataFrame(valid_records)
del valid_records 

In [249]:
df.head()

,timestamp,service,level,message,request_id,trace_id
0,2026-07-27T00:02:47Z,notification-worker,ERROR,ERR SMTPConnRefused host=mail-gw,req-65568711,NaN
1,2026-07-27T00:12:06Z,auth-service,INFO,Session created uid=u2746,req-72350830,NaN
2,2026-07-27T00:13:20Z,payment-api,WARN,Retry 1/3 calling notification-worker,req-22315507,NaN
3,2026-07-27T00:20:17Z,notification-worker,INFO,SMS sent uid=u1132,req-27740248,NaN
4,2026-07-27T00:27:38Z,auth-service,WARN,Slow login 900ms uid=u7882,req-95306788,NaN


In [250]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2905 entries, 0 to 2904
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   timestamp   2905 non-null   str  
 1   service     2905 non-null   str  
 2   level       2887 non-null   str  
 3   message     2905 non-null   str  
 4   request_id  2905 non-null   str  
 5   trace_id    1199 non-null   str  
dtypes: str(6)
memory usage: 136.3 KB


In [251]:
non_null_counts = df.notnull().sum()
null_counts = df.isnull().sum()
null_percentage = (null_counts / len(df)) * 100

summary_df = pd.DataFrame({
	"Non-null Count": non_null_counts,
	"Null Count": null_counts,
	"Null Percentage (%)": null_percentage.round(2)
})

print(summary_df)

            Non-null Count  Null Count  Null Percentage (%)
timestamp             2905           0                 0.00
service               2905           0                 0.00
level                 2887          18                 0.62
message               2905           0                 0.00
request_id            2905           0                 0.00
trace_id              1199        1706                58.73


There are many duplicated records

In [252]:
duplicated_df =df[df.duplicated(keep=False)]

display(duplicated_df)
print(len(duplicated_df), "duplicated records found.")

,timestamp,service,level,message,request_id,trace_id
56,2026-07-27T03:51:26Z,batch-report,WARN,Report row mismatch expected=1061 got=1082,req-84892608,NaN
57,2026-07-27T03:51:26Z,batch-report,WARN,Report row mismatch expected=1061 got=1082,req-84892608,NaN
89,2026-07-27T05:52:54Z,auth-service,INFO,Token refreshed uid=u9837,req-25014631,NaN
90,2026-07-27T05:52:54Z,auth-service,INFO,Token refreshed uid=u9837,req-25014631,NaN
208,2026-07-27T12:48:15Z,notification-worker,INFO,Email sent uid=u3328,req-33562830,NaN
209,2026-07-27T12:48:15Z,notification-worker,INFO,Email sent uid=u3328,req-33562830,NaN
320,2026-07-27T19:44:26Z,auth-service,INFO,Session created uid=u1998,req-77491435,NaN
321,2026-07-27T19:44:26Z,auth-service,INFO,Session created uid=u1998,req-77491435,NaN
371,2026-07-27T22:23:19Z,batch-report,INFO,Daily report job finished rows=851,req-69920305,NaN
372,2026-07-27T22:23:19Z,batch-report,INFO,Daily report job finished rows=851,req-69920305,NaN


56 duplicated records found.


In [253]:
first_duplicated_df =df[df.duplicated(keep='first')]

display(first_duplicated_df)
print(len(first_duplicated_df), "duplicated records found.")


,timestamp,service,level,message,request_id,trace_id
57,2026-07-27T03:51:26Z,batch-report,WARN,Report row mismatch expected=1061 got=1082,req-84892608,NaN
90,2026-07-27T05:52:54Z,auth-service,INFO,Token refreshed uid=u9837,req-25014631,NaN
209,2026-07-27T12:48:15Z,notification-worker,INFO,Email sent uid=u3328,req-33562830,NaN
321,2026-07-27T19:44:26Z,auth-service,INFO,Session created uid=u1998,req-77491435,NaN
372,2026-07-27T22:23:19Z,batch-report,INFO,Daily report job finished rows=851,req-69920305,NaN
474,2026-07-28T04:54:18Z,payment-api,INFO,Payment processed txn=t141094 amount=120000,req-79342339,NaN
535,2026-07-28T10:14:59Z,auth-service,INFO,Token refreshed uid=u5872,req-75893962,NaN
766,2026-07-28T23:38:39Z,batch-report,INFO,Daily report job finished rows=1163,req-75050252,NaN
824,2026-07-29T01:39:57Z,batch-report,INFO,Daily report job finished rows=1110,req-92220752,NaN
842,2026-07-29T02:44:06Z,notification-worker,INFO,Email sent uid=u7013,req-66736822,NaN


28 duplicated records found.


## 1. timestamp

In [254]:
def detect_ts_format(x):
    x = str(x)

    if re.fullmatch(r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}Z", x):
        return "UTC_Z"                     # 2026-07-27T08:30:00Z

    elif re.fullmatch(r"\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}[+-]\d{2}:\d{2}", x):
        return "ISO_timezone_offset"        # 2026-07-27T07:58:15+07:00

    elif re.fullmatch(r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}", x):
        return "datetime_space"             # 2026-07-27 08:30:00

    elif re.fullmatch(r"\d{4}-\d{2}-\d{2}", x):
        return "date_only"                  # 2026-07-27

    else:
        return "unknown/invalid"

df["timestamp_format"] = df["timestamp"].apply(detect_ts_format)

print(df["timestamp_format"].value_counts())

timestamp_format
UTC_Z                  2289
ISO_timezone_offset     596
unknown/invalid          20
Name: count, dtype: int64


In [255]:
df[df['timestamp_format'] == 'unknown/invalid']

,timestamp,service,level,message,request_id,trace_id,timestamp_format
33,not-a-date,auth-service,WARN,Clock sync failed,req-32170750,NaN,unknown/invalid
228,not-a-date,auth-service,WARN,Clock sync failed,req-80542438,trace-2840524582,unknown/invalid
483,not-a-date,auth-service,WARN,Clock sync failed,req-64770806,trace-3597287435,unknown/invalid
502,not-a-date,batch-report,WARN,Clock sync failed,req-89141563,trace-1842573576,unknown/invalid
553,not-a-date,notification-worker,WARN,Clock sync failed,req-95756894,trace-1839947796,unknown/invalid
696,not-a-date,payment-api,WARN,Clock sync failed,req-46941691,NaN,unknown/invalid
1029,not-a-date,batch-report,WARN,Clock sync failed,req-58571652,trace-8069263805,unknown/invalid
1191,not-a-date,batch-report,WARN,Clock sync failed,req-82179030,trace-5297869651,unknown/invalid
1243,not-a-date,web-portal,WARN,Clock sync failed,req-14251180,trace-2529141801,unknown/invalid
1349,not-a-date,web-portal,WARN,Clock sync failed,req-62013608,NaN,unknown/invalid


In [256]:
df["timezone_offset"] = df["timestamp"].str.extract(
    r"([+-]\d{2}:\d{2})$"
)

print(df["timezone_offset"].value_counts())

timezone_offset
+07:00    596
Name: count, dtype: int64


In [257]:
display(df[df["timezone_offset"].notna()])

,timestamp,service,level,message,request_id,trace_id,timestamp_format,timezone_offset
119,2026-07-27T07:58:15+07:00,web-portal,INFO,Request completed path=/report in 554ms,req-20952896,NaN,ISO_timezone_offset,+07:00
125,2026-07-27T08:23:40+07:00,web-portal,INFO,Request completed path=/home in 752ms,req-70210270,NaN,ISO_timezone_offset,+07:00
126,2026-07-27T08:25:30+07:00,web-portal,INFO,Request completed path=/home in 489ms,req-31306555,NaN,ISO_timezone_offset,+07:00
133,2026-07-27T08:53:59+07:00,web-portal,INFO,Request completed path=/report in 768ms,req-48260510,NaN,ISO_timezone_offset,+07:00
137,2026-07-27T09:10:52+07:00,web-portal,INFO,Request completed path=/home in 535ms,req-91424562,NaN,ISO_timezone_offset,+07:00
...,...,...,...,...,...,...,...,...
2900,2026-08-03T06:15:30+07:00,web-portal,WARN,Response time 2100ms path=/report,req-45353565,trace-8774772376,ISO_timezone_offset,+07:00
2901,2026-08-03T06:30:10+07:00,web-portal,ERROR,ERR HTTP 502 upstream=payment-api path=/checkout,req-99609973,trace-7354458743,ISO_timezone_offset,+07:00
2902,2026-08-03T06:43:40+07:00,web-portal,INFO,Request completed path=/report in 284ms,req-87206285,trace-5698353763,ISO_timezone_offset,+07:00
2903,2026-08-03T06:55:40+07:00,web-portal,INFO,Request completed path=/home in 871ms,req-72913088,trace-5937414387,ISO_timezone_offset,+07:00


---

In [258]:
df['event_date_utc'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce').dt.date
display(df['event_date_utc'])

print(type(df['event_date_utc'].iloc[0])) # first row

0       2026-07-27
1       2026-07-27
2       2026-07-27
3       2026-07-27
4       2026-07-27
           ...    
2900    2026-08-02
2901    2026-08-02
2902    2026-08-02
2903    2026-08-02
2904    2026-08-02
Name: event_date_utc, Length: 2905, dtype: object

<class 'datetime.date'>


## 2. service

In [259]:
df['service'].unique()

<StringArray>
['notification-worker',        'auth-service',         'payment-api',
        'batch-report',          'web-portal']
Length: 5, dtype: str

## 3. level

In [260]:
df['level'].unique()

<StringArray>
['ERROR', 'INFO', 'WARN', nan]
Length: 4, dtype: str

18 missing values out of 2905 records

In [261]:
df[df['level'].isna()]

,timestamp,service,level,message,request_id,trace_id,timestamp_format,timezone_offset,event_date_utc
10,2026-07-30T12:07:36Z,notification-worker,NaN,Heartbeat ok,req-48936328,NaN,UTC_Z,NaN,2026-07-30
94,2026-07-30T18:23:58Z,payment-api,NaN,Heartbeat ok,req-30906603,NaN,UTC_Z,NaN,2026-07-30
213,2026-07-30T13:53:09Z,payment-api,NaN,Heartbeat ok,req-89098923,NaN,UTC_Z,NaN,2026-07-30
300,2026-07-31T18:38:08Z,auth-service,NaN,Heartbeat ok,req-18963036,trace-4082960786,UTC_Z,NaN,2026-07-31
697,2026-08-01T01:47:50Z,payment-api,NaN,Heartbeat ok,req-21881015,trace-7088048157,UTC_Z,NaN,2026-08-01
703,2026-08-01T07:53:41+07:00,web-portal,NaN,Heartbeat ok,req-71725422,trace-9264884678,ISO_timezone_offset,+07:00,2026-08-01
830,2026-08-02T01:19:22Z,auth-service,NaN,Heartbeat ok,req-11122998,trace-7153186561,UTC_Z,NaN,2026-08-02
1038,2026-07-29T03:14:59Z,batch-report,NaN,Heartbeat ok,req-22252693,NaN,UTC_Z,NaN,2026-07-29
1277,2026-07-28T18:01:01Z,auth-service,NaN,Heartbeat ok,req-29057175,NaN,UTC_Z,NaN,2026-07-28
1316,2026-07-27T23:43:39Z,batch-report,NaN,Heartbeat ok,req-41470269,NaN,UTC_Z,NaN,2026-07-27


## 4. message

In [262]:
df['message'].head(10)

0            ERR SMTPConnRefused host=mail-gw
1                   Session created uid=u2746
2       Retry 1/3 calling notification-worker
3                          SMS sent uid=u1132
4                  Slow login 900ms uid=u7882
5                   Session created uid=u2528
6                        Email sent uid=u2106
7                    Daily report job started
8    Report row mismatch expected=843 got=759
9                          SMS sent uid=u5942
Name: message, dtype: str

In [263]:
df['message'].unique()

<StringArray>
[        'ERR SMTPConnRefused host=mail-gw',
                'Session created uid=u2746',
    'Retry 1/3 calling notification-worker',
                       'SMS sent uid=u1132',
               'Slow login 900ms uid=u7882',
                'Session created uid=u2528',
                     'Email sent uid=u2106',
                 'Daily report job started',
 'Report row mismatch expected=843 got=759',
                       'SMS sent uid=u5942',
 ...
  'Request completed path=/report in 167ms',
  'Request completed path=/report in 643ms',
    'Request completed path=/home in 753ms',
    'Request completed path=/home in 178ms',
  'Request completed path=/report in 332ms',
    'Request completed path=/home in 795ms',
    'Request completed path=/home in 868ms',
  'Request completed path=/report in 284ms',
    'Request completed path=/home in 871ms',
  'Request completed path=/report in 109ms']
Length: 2153, dtype: str

High cardinality: 2,153 distinct strings / 2,905 rows, because messages embed dynamic values such as user IDs, transaction IDs, amounts, paths, latency, row counts, etc

## 5. request_id

In [264]:
df['request_id'].head(10)

0    req-65568711
1    req-72350830
2    req-22315507
3    req-27740248
4    req-95306788
5    req-38688676
6    req-63415303
7    req-67790557
8    req-56751880
9    req-42123815
Name: request_id, dtype: str

Check to see all values match `req-########`

In [265]:
pattern = r"req-\d{8}"

# Rows that do not match the pattern
df[~df['request_id'].str.fullmatch(pattern, na=False)]

,timestamp,service,level,message,request_id,trace_id,timestamp_format,timezone_offset,event_date_utc


Since the `trace_id` column contains many missing values, so I dont treat it as a primary column. Instead, I use `request_id`

In [266]:
duplicated_request_df = df[df.duplicated(subset=['request_id'], keep='first')]
display(duplicated_request_df)

print(len(duplicated_request_df), "duplicated request_id records found.")

,timestamp,service,level,message,request_id,trace_id,timestamp_format,timezone_offset,event_date_utc
57,2026-07-27T03:51:26Z,batch-report,WARN,Report row mismatch expected=1061 got=1082,req-84892608,NaN,UTC_Z,NaN,2026-07-27
90,2026-07-27T05:52:54Z,auth-service,INFO,Token refreshed uid=u9837,req-25014631,NaN,UTC_Z,NaN,2026-07-27
209,2026-07-27T12:48:15Z,notification-worker,INFO,Email sent uid=u3328,req-33562830,NaN,UTC_Z,NaN,2026-07-27
321,2026-07-27T19:44:26Z,auth-service,INFO,Session created uid=u1998,req-77491435,NaN,UTC_Z,NaN,2026-07-27
372,2026-07-27T22:23:19Z,batch-report,INFO,Daily report job finished rows=851,req-69920305,NaN,UTC_Z,NaN,2026-07-27
474,2026-07-28T04:54:18Z,payment-api,INFO,Payment processed txn=t141094 amount=120000,req-79342339,NaN,UTC_Z,NaN,2026-07-28
535,2026-07-28T10:14:59Z,auth-service,INFO,Token refreshed uid=u5872,req-75893962,NaN,UTC_Z,NaN,2026-07-28
766,2026-07-28T23:38:39Z,batch-report,INFO,Daily report job finished rows=1163,req-75050252,NaN,UTC_Z,NaN,2026-07-28
824,2026-07-29T01:39:57Z,batch-report,INFO,Daily report job finished rows=1110,req-92220752,NaN,UTC_Z,NaN,2026-07-29
842,2026-07-29T02:44:06Z,notification-worker,INFO,Email sent uid=u7013,req-66736822,NaN,UTC_Z,NaN,2026-07-29


28 duplicated request_id records found.


## 6. trace_id

As calculated earlier: 
- Non-null counts: 1199 records
- Null counts: 1706 records

In [267]:
df[~df['trace_id'].isna()]

,timestamp,service,level,message,request_id,trace_id,timestamp_format,timezone_offset,event_date_utc
228,not-a-date,auth-service,WARN,Clock sync failed,req-80542438,trace-2840524582,unknown/invalid,NaN,NaT
300,2026-07-31T18:38:08Z,auth-service,NaN,Heartbeat ok,req-18963036,trace-4082960786,UTC_Z,NaN,2026-07-31
483,not-a-date,auth-service,WARN,Clock sync failed,req-64770806,trace-3597287435,unknown/invalid,NaN,NaT
502,not-a-date,batch-report,WARN,Clock sync failed,req-89141563,trace-1842573576,unknown/invalid,NaN,NaT
553,not-a-date,notification-worker,WARN,Clock sync failed,req-95756894,trace-1839947796,unknown/invalid,NaN,NaT
...,...,...,...,...,...,...,...,...,...
2900,2026-08-03T06:15:30+07:00,web-portal,WARN,Response time 2100ms path=/report,req-45353565,trace-8774772376,ISO_timezone_offset,+07:00,2026-08-02
2901,2026-08-03T06:30:10+07:00,web-portal,ERROR,ERR HTTP 502 upstream=payment-api path=/checkout,req-99609973,trace-7354458743,ISO_timezone_offset,+07:00,2026-08-02
2902,2026-08-03T06:43:40+07:00,web-portal,INFO,Request completed path=/report in 284ms,req-87206285,trace-5698353763,ISO_timezone_offset,+07:00,2026-08-02
2903,2026-08-03T06:55:40+07:00,web-portal,INFO,Request completed path=/home in 871ms,req-72913088,trace-5937414387,ISO_timezone_offset,+07:00,2026-08-02


In [268]:
pattern = r"trace-\d{10}"

# Rows that do not match the pattern (exclude NaN values)
df[~df['trace_id'].str.fullmatch(pattern, na=True)]

,timestamp,service,level,message,request_id,trace_id,timestamp_format,timezone_offset,event_date_utc


Let's check when the `trace_id` starts existing

In [271]:
missing_trace_df = df[df['trace_id'].isna()]
missing_trace_df[missing_trace_df['event_date_utc'].notna()].sort_values(by='event_date_utc', ascending=True)

,timestamp,service,level,message,request_id,trace_id,timestamp_format,timezone_offset,event_date_utc
0,2026-07-27T00:02:47Z,notification-worker,ERROR,ERR SMTPConnRefused host=mail-gw,req-65568711,NaN,UTC_Z,NaN,2026-07-27
279,2026-07-27T16:58:32Z,batch-report,INFO,Daily report job finished rows=933,req-76743428,NaN,UTC_Z,NaN,2026-07-27
278,2026-07-27T16:46:41Z,batch-report,INFO,Daily report job finished rows=871,req-75540707,NaN,UTC_Z,NaN,2026-07-27
277,2026-07-27T16:43:56Z,payment-api,INFO,Balance check ok uid=u5183,req-83363160,NaN,UTC_Z,NaN,2026-07-27
276,2026-07-27T16:40:03+07:00,web-portal,WARN,Response time 2100ms path=/report,req-94426338,NaN,ISO_timezone_offset,+07:00,2026-07-27
...,...,...,...,...,...,...,...,...,...
1359,2026-07-30T08:20:59Z,payment-api,ERROR,ERR PaymentDeclined txn=t940208 code=51,req-84215849,NaN,UTC_Z,NaN,2026-07-30
1358,2026-07-30T08:18:53Z,notification-worker,INFO,SMS sent uid=u7091,req-43040802,NaN,UTC_Z,NaN,2026-07-30
1357,2026-07-30T08:18:42Z,payment-api,INFO,Balance check ok uid=u9606,req-23420761,NaN,UTC_Z,NaN,2026-07-30
1387,2026-07-30T09:44:33Z,payment-api,INFO,Payment processed txn=t604967 amount=120000,req-97499829,NaN,UTC_Z,NaN,2026-07-30


In [281]:
non_missing_trace_df = df[df['trace_id'].notna() & df['event_date_utc'].notna()]
non_missing_trace_df.sort_values(by='event_date_utc', ascending=True)

,timestamp,service,level,message,request_id,trace_id,timestamp_format,timezone_offset,event_date_utc
300,2026-07-31T18:38:08Z,auth-service,NaN,Heartbeat ok,req-18963036,trace-4082960786,UTC_Z,NaN,2026-07-31
1984,2026-07-31T17:11:12+07:00,web-portal,WARN,Response time 2100ms path=/report,req-59442025,trace-3228001003,ISO_timezone_offset,+07:00,2026-07-31
1983,2026-07-31T17:05:15Z,payment-api,INFO,Payment processed txn=t195407 amount=990000,req-43768232,trace-7997023025,UTC_Z,NaN,2026-07-31
1982,2026-07-31T16:54:28Z,auth-service,INFO,Token refreshed uid=u8408,req-13285625,trace-4751060216,UTC_Z,NaN,2026-07-31
1981,2026-07-31T16:52:13Z,notification-worker,WARN,Queue depth high depth=1594,req-15789440,trace-7824023067,UTC_Z,NaN,2026-07-31
...,...,...,...,...,...,...,...,...,...
2631,2026-08-02T08:26:55Z,batch-report,INFO,Daily report job finished rows=1152,req-64262105,trace-4638106414,UTC_Z,NaN,2026-08-02
2630,2026-08-02T08:19:18Z,auth-service,INFO,Token refreshed uid=u1433,req-46048156,trace-5885218538,UTC_Z,NaN,2026-08-02
2629,2026-08-02T08:18:39Z,auth-service,WARN,Slow login 900ms uid=u8586,req-67124503,trace-3830532859,UTC_Z,NaN,2026-08-02
2639,2026-08-02T08:45:34Z,notification-worker,INFO,SMS sent uid=u4826,req-21139313,trace-7008973602,UTC_Z,NaN,2026-08-02


The missingness is highly structured: for valid timestamps, events on Jul 27–30 have no trace ID, while Jul 31–Aug 2 have one, suggesting a source/schema evolution rather than random bad data. Here are some duplicated trace IDs

In [287]:
dup_non_missing_trace_df = non_missing_trace_df[non_missing_trace_df.duplicated(subset=['trace_id'], keep=False)]
display(dup_non_missing_trace_df)

print(len(dup_non_missing_trace_df), "duplicated trace_id records found.")

,timestamp,service,level,message,request_id,trace_id,timestamp_format,timezone_offset,event_date_utc
1753,2026-07-31T02:11:57Z,auth-service,INFO,Token refreshed uid=u8046,req-52734662,trace-9872881926,UTC_Z,NaN,2026-07-31
1754,2026-07-31T02:11:57Z,auth-service,INFO,Token refreshed uid=u8046,req-52734662,trace-9872881926,UTC_Z,NaN,2026-07-31
1914,2026-07-31T13:13:16+07:00,web-portal,WARN,Response time 2100ms path=/report,req-42576819,trace-9072159605,ISO_timezone_offset,+07:00,2026-07-31
1915,2026-07-31T13:13:16+07:00,web-portal,WARN,Response time 2100ms path=/report,req-42576819,trace-9072159605,ISO_timezone_offset,+07:00,2026-07-31
1925,2026-07-31T13:55:22+07:00,web-portal,INFO,Request completed path=/home in 802ms,req-84820714,trace-8605695669,ISO_timezone_offset,+07:00,2026-07-31
1926,2026-07-31T13:55:22+07:00,web-portal,INFO,Request completed path=/home in 802ms,req-84820714,trace-8605695669,ISO_timezone_offset,+07:00,2026-07-31
1993,2026-07-31T17:43:22Z,payment-api,INFO,Balance check ok uid=u7541,req-65807734,trace-5121529498,UTC_Z,NaN,2026-07-31
1994,2026-07-31T17:43:22Z,payment-api,INFO,Balance check ok uid=u7541,req-65807734,trace-5121529498,UTC_Z,NaN,2026-07-31
2049,2026-07-31T21:43:41+07:00,web-portal,INFO,Request completed path=/home in 315ms,req-66401535,trace-7894521683,ISO_timezone_offset,+07:00,2026-07-31
2050,2026-07-31T21:43:41+07:00,web-portal,INFO,Request completed path=/home in 315ms,req-66401535,trace-7894521683,ISO_timezone_offset,+07:00,2026-07-31


22 duplicated trace_id records found.
